In [1]:
from scipy.interpolate import BSpline

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from dataclasses import dataclass, field
from typing import List

from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

from sklearn.metrics import r2_score, mean_squared_error,mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import pickle
import os
import pandas as pd
import re
import joblib

In [2]:
# r for 22 sections
geometry_baseline_r = [0.0201254, 0.0252054, 0.0315579, 0.0379079, 0.0442579, 0.0506079, 0.0569579,
 0.0633079, 0.0696579, 0.0760079, 0.0823586, 0.0887148, 0.0950769, 0.1014424,
 0.107804,  0.1141792, 0.1173761, 0.1205864, 0.1238169, 0.1255,    0.12625,
 0.127]

# generate_bspline_basis_custom
### param : control_points(size: 1 * 8)
### return : matrices（3 * 22）--（r/R, chord, twist）


In [3]:
def generate_section(n_control_points):
    chord_points = n_control_points[0:4]
    twist_points = n_control_points[4:]
    degree = 3
    knots = np.concatenate(([0] * degree, [0.3984874, 0.89904882], [1] * degree))
    twist_knots = np.concatenate(([0] * degree, [0.2, 0.89904882], [1] * degree))

    chord_bspline = BSpline(knots, chord_points, degree)
    twist_bspline = BSpline(twist_knots, twist_points, degree)
    x_norm = [x / 0.127 for x in geometry_baseline_r]
    chord_spline = chord_bspline(x_norm)
    twist_spline = twist_bspline(x_norm)

    propeller_geometry = np.array([x_norm, chord_spline, twist_spline]).T
    return propeller_geometry


# Propeller Modeling Net

## congig for model

In [ ]:
class PropellerPredictor(nn.Module):
    def __init__(self, input_dim=47, hidden_dims=[128, 64, 32], output_dim=4):
        super().__init__()
        layers = []
        dims = [input_dim] + hidden_dims
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            layers.append(nn.BatchNorm1d(dims[i + 1]))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(dims[-1], output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# traing

## create a tensorboard for recording the training process

In [5]:
log_dir = f'runs/propeller_predictor_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
writer = SummaryWriter(log_dir=log_dir)


## deal_data for trainging LFM model

In [6]:
# data_path = './data_for_train'
# data_files = os.listdir(data_path)
# all_data = {}
# for file in data_files:
#     if file.endswith('.pkl') and 'data' in str(file):
#         with open(os.path.join(data_path, file), 'rb') as f:
#             data = pickle.load(f)
#             all_data.update(data)

### data_structure
- keys:geometry_num
  - values: 
    - keys: RMP****_Wind**_Angle**
      - values: Time,Thrust,Power,Torch,Trust_y,Thrust_z -- dataframes

In [7]:
# columns_runningconditon = ['geometry_num', 'RPM', 'WIND', 'ANGLE',]
# chord_columns = [f'chord_{i}' for i in range(22)]
# twist_columns = [f'twist_{i}' for i in range(22)]
# output_columns = ['Power', 'Fx', 'Fy', 'Fz', 'Torque']
# columns = columns_runningconditon + chord_columns + twist_columns + output_columns
# data_total = pd.DataFrame(columns=columns) # create an empty dataframe to store all data

# # iteration all_data
# for geo_idx, (key, value) in enumerate(all_data.items()):
#     chord = None
#     twist = None
#     data_list = []

#     for key_, value_ in value.items():
#         # store the geometry
#         if 'geometry' in key_:
#             chord = value_[:, 1]
#             twist = value_[:, 2]
        
#         elif 'RPM' in key_:
#             # store the running condition
#             numbers = re.findall(r'\d+(?:\.\d+)?', key_) 
#             RPM, WIND, ANGLE = map(float, numbers)
#             data_one = pd.DataFrame(columns=columns)
#             data_one.at[0, 'geometry_num'] = geo_idx
#             data_one.at[0, 'RPM'] = RPM
#             data_one.at[0, 'WIND'] = WIND
#             data_one.at[0, 'ANGLE'] = ANGLE
#             # store the output data
#             data_one.at[0, 'Fx'] = -value_['Thrust'].mean() # (because of the direction of thrust)
#             data_one.at[0, 'Fy'] = value_['Trust_y'].mean()
#             data_one.at[0, 'Fz'] = value_['Trust_z'].mean()
#             data_one.at[0, 'Power'] = -value_['Power'].mean() # (because of the direction of thrust)
#             data_one.at[0, 'Torque'] = -value_['Torque'].mean() # (because of the direction of thrust)

#             # ignore the chord and twist data because the order is not sure
#             data_list.append(data_one)

#     # geometry data has been stored, now store the chord and twist data 
#     if chord is not None and twist is not None:
#         for data_one in data_list:
#             for i in range(22):
#                 data_one.at[0, f'chord_{i}'] = chord[i]
#                 data_one.at[0, f'twist_{i}'] = twist[i]
#             data_total = pd.concat([data_total, data_one], ignore_index=True)
# # save the data
# data_path = os.path.join('./data_for_train', 'dealed_data.xlsx')
# data_total.to_excel(data_path)

#### standard the data

In [8]:
# ==============================
# 加载数据
# ==============================
# load the data from excel
df = pd.read_excel('./data_for_train/dealed_data.xlsx')

# ==============================
# 定义输入输出列
# ==============================
# define the columns we need
columns_runningconditon = ['RPM', 'WIND', 'ANGLE',]
chord_columns = [f'chord_{i}' for i in range(22)]
twist_columns = [f'twist_{i}' for i in range(22)]
output_columns = ['Fx', 'Fy', 'Fz', 'Torque']


# ==============================
# 切分数据集
# ==============================
# split the data into Input set and Output set
X = df[columns_runningconditon + chord_columns + twist_columns]
Y = df[output_columns]
# split the data into training set and testing set
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# ==============================
# 初始化标准化器
# ==============================
# initialize the scaler
scaler_runningconditon = MinMaxScaler()
scaler_chord = MinMaxScaler()
scaler_twist = MinMaxScaler()
scaler_Fx = MinMaxScaler()
scaler_Fy = MinMaxScaler()
scaler_Fz = MinMaxScaler()
scaler_Torque = MinMaxScaler()

# ==============================
# 对训练集进行标准化
# ==============================
# fit the scaler on the training set
X_train_runningconditon_scaled = scaler_runningconditon.fit_transform(X_train[columns_runningconditon])
X_train_chord_scaled = scaler_chord.fit_transform(X_train[chord_columns]) 
X_train_twist_scaled = scaler_twist.fit_transform(X_train[twist_columns]) 
Y_train_Fx_scaled = scaler_Fx.fit_transform(y_train[['Fx']])
Y_train_Fy_scaled = scaler_Fy.fit_transform(y_train[['Fy']])
Y_train_Fz_scaled = scaler_Fz.fit_transform(y_train[['Fz']])
Y_train_Torque_scaled = scaler_Torque.fit_transform(y_train[['Torque']])

# ==============================
# 对测试集进行标准化
# ==============================
# transform the testing set
X_test_runningconditon_scaled = scaler_runningconditon.transform(X_test[columns_runningconditon])
X_test_chord_scaled = scaler_chord.transform(X_test[chord_columns]) 
X_test_twist_scaled = scaler_twist.transform(X_test[twist_columns]) 
Y_test_Fx_scaled = scaler_Fx.transform(y_test[['Fx']])
Y_test_Fy_scaled = scaler_Fy.transform(y_test[['Fy']])
Y_test_Fz_scaled = scaler_Fz.transform(y_test[['Fz']])
Y_test_Torque_scaled = scaler_Torque.transform(y_test[['Torque']])

# ==============================
# 合并标准化后的数据
# ==============================
# Results after merger normalization
X_train_scaled = np.hstack([X_train_runningconditon_scaled, X_train_chord_scaled, X_train_twist_scaled])
X_test_scaled = np.hstack([X_test_runningconditon_scaled, X_test_chord_scaled, X_test_twist_scaled])
Y_train_scaled = np.hstack([Y_train_Fx_scaled, Y_train_Fy_scaled, Y_train_Fz_scaled, Y_train_Torque_scaled])
Y_test_scaled = np.hstack([Y_test_Fx_scaled, Y_test_Fy_scaled, Y_test_Fz_scaled, Y_test_Torque_scaled])

# ==============================
# 转换为 DataFrame
# ==============================
# transform into dataframe
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=columns_runningconditon + chord_columns + twist_columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=columns_runningconditon + chord_columns + twist_columns)
Y_train_scaled_df = pd.DataFrame(Y_train_scaled, columns=output_columns)
Y_test_scaled_df = pd.DataFrame(Y_test_scaled, columns=output_columns)

# ==============================
# 保存数据到文件
# ==============================
# save the data into pickle file
X_train_scaled_df.to_pickle(os.path.join('./data_for_train', 'X_train_scaled'))
X_test_scaled_df.to_pickle(os.path.join('./data_for_train', 'X_test_scaled'))
Y_train_scaled_df.to_pickle(os.path.join('./data_for_train', 'y_train_scaled'))
Y_test_scaled_df.to_pickle(os.path.join('./data_for_train', 'y_test_scaled'))

# ==============================
# 保存所有标准化器
# ==============================
# save the scaler
joblib.dump(scaler_runningconditon, './data_for_train/runningconditon_scaler.pkl')
joblib.dump(scaler_chord, './data_for_train/chord_scaler.pkl')
joblib.dump(scaler_twist, './data_for_train/twist_scaler.pkl')
joblib.dump(scaler_Fx, './data_for_train/Fx_scaler.pkl')
joblib.dump(scaler_Fy, './data_for_train/Fy_scaler.pkl')
joblib.dump(scaler_Fz, './data_for_train/Fz_scaler.pkl')
joblib.dump(scaler_Torque, './data_for_train/Torque_scaler.pkl')

['./data_for_train/Torque_scaler.pkl']

## Hyperparameters

In [9]:
# ==============================
# 确定超参数
# ==============================
# define the model
l2_lambda = 1e-4
device = torch.device("cpu")
model = PropellerPredictor().to(device)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4,  weight_decay=1e-4)
batch_size = 64
epochs = 10000


## traing

### eval for tensorboard recording

In [10]:
# ==============================
# 评价函数
# ==============================
def evaluate_metrics(y_true, y_pred):
    y_true_np = y_true.detach().cpu().numpy()
    y_pred_np = y_pred.detach().cpu().numpy()
    mae = mean_absolute_error(y_true_np, y_pred_np)
    r2 = r2_score(y_true_np, y_pred_np)
    rel_error = np.abs((y_pred_np - y_true_np) / (np.abs(y_true_np) + 1e-8))
    return mae, r2, rel_error

### load data

In [11]:
# ==============================
# 加载数据，均为预处理数据(标准化)
# ==============================
X_train = torch.tensor(pd.read_pickle('./data_for_train/X_train_scaled').values, dtype=torch.float32)
X_test = torch.tensor(pd.read_pickle('./data_for_train/X_test_scaled').values, dtype=torch.float32)
y_train = torch.tensor(pd.read_pickle('./data_for_train/y_train_scaled').values, dtype=torch.float32)
y_test = torch.tensor(pd.read_pickle('./data_for_train/y_test_scaled').values, dtype=torch.float32)

# ==============================
# 分批次加载数据集
# ==============================
# dataloader setting
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)

In [12]:
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = loss_fn(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_X.size(0)
    model.eval()
    with torch.no_grad():
        train_outputs = model(X_train)
        test_outputs = model(X_test)

        # record loss
        writer.add_scalar("Loss/Train", loss_fn(train_outputs, y_train).item(), epoch)
        writer.add_scalar("Loss/Test", loss_fn(test_outputs, y_test).item(), epoch)

        # record MAE, R², relative error
        train_mae, train_r2, train_rel = evaluate_metrics(y_train, train_outputs)
        test_mae, test_r2, test_rel = evaluate_metrics(y_test, test_outputs)
        writer.add_scalar("MAE/Train", train_mae, epoch)
        writer.add_scalar("MAE/Test", test_mae, epoch)
        writer.add_scalar("R2/Train", train_r2, epoch)
        writer.add_scalar("R2/Test", test_r2, epoch)
        writer.add_scalar("RelativeError/Train", train_rel.mean(), epoch)
        writer.add_scalar("RelativeError/Test", test_rel.mean(), epoch)
        output_names = ['Power', 'Fx', 'Fy', 'Fz']
        for i, name in enumerate(output_names):
            writer.add_scalar(f'RelativeError/Train/{name}', np.mean(train_rel[:, i]), epoch)
            writer.add_scalar(f'RelativeError/Test/{name}', np.mean(test_rel[:, i]), epoch)

    avg_loss = total_loss / len(train_loader.dataset)
    writer.add_scalar("Loss/train", avg_loss, epoch)
    
    # print information for each 500
    if epoch % 500 == 0:
        print(f"[Epoch {epoch}] Loss: {avg_loss:.6f}")
        print(f"Test MAE: {(test_mae + train_mae) / 2:.6f}")
        print(f"Test R²: {(test_r2 + train_r2) / 2:.6f}")
    

[Epoch 0] Loss: 0.129072
Test MAE: 0.226973
Test R²: -0.810709
[Epoch 500] Loss: 0.000483
Test MAE: 0.009752
Test R²: 0.996168
[Epoch 1000] Loss: 0.000440
Test MAE: 0.010499
Test R²: 0.994646
[Epoch 1500] Loss: 0.000343
Test MAE: 0.010566
Test R²: 0.995038
[Epoch 2000] Loss: 0.000338
Test MAE: 0.009713
Test R²: 0.995462
[Epoch 2500] Loss: 0.000310
Test MAE: 0.008460
Test R²: 0.996926
[Epoch 3000] Loss: 0.000302
Test MAE: 0.009509
Test R²: 0.995928
[Epoch 3500] Loss: 0.000325
Test MAE: 0.008207
Test R²: 0.996749
[Epoch 4000] Loss: 0.000244
Test MAE: 0.007574
Test R²: 0.997402
[Epoch 4500] Loss: 0.000261
Test MAE: 0.007981
Test R²: 0.997223
[Epoch 5000] Loss: 0.000276
Test MAE: 0.008278
Test R²: 0.996766
[Epoch 5500] Loss: 0.000320
Test MAE: 0.007756
Test R²: 0.997277
[Epoch 6000] Loss: 0.000327
Test MAE: 0.010548
Test R²: 0.995142
[Epoch 6500] Loss: 0.000684
Test MAE: 0.022230
Test R²: 0.980016
[Epoch 7000] Loss: 0.000316
Test MAE: 0.009136
Test R²: 0.996102
[Epoch 7500] Loss: 0.000293


In [13]:
## save the model to disk

In [14]:
os.makedirs("trained_models", exist_ok=True)
torch.save(model.state_dict(), "trained_models/propeller_predictor_cpu.pth")
writer.close()

# prediction

## load the scaler

In [15]:
scaler_Fx = joblib.load('./data_for_train/Fx_scaler.pkl')
scaler_Fy = joblib.load('./data_for_train/Fy_scaler.pkl')
scaler_Fz = joblib.load('./data_for_train/Fz_scaler.pkl')
scaler_Torque = joblib.load('./data_for_train/Torque_scaler.pkl')

In [ ]:
X_test = pd.read_pickle('./data_for_train/X_test_scaled')
y_test = pd.read_pickle('./data_for_train/y_test_scaled')
model.load_state_dict(torch.load('trained_models/propeller_predictor_cpu.pth'))  # 请根据你保存的模型文件名修改
model.eval()

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32).to(device)
with torch.no_grad():
    y_pred_tensor = model(X_test_tensor).cpu()
# pred results
y_pred = y_pred_tensor.numpy()
y_pred_original = np.hstack([
    scaler_Fx.inverse_transform(y_pred[:, [0]]),
    scaler_Fy.inverse_transform(y_pred[:, [1]]),
    scaler_Fz.inverse_transform(y_pred[:, [2]]),
    scaler_Torque.inverse_transform(y_pred[:, [3]]),
])
df_result = pd.DataFrame(y_pred_original, columns=['Fx_pred', 'Fy_pred', 'Fz_pred', 'Torque_pred'])

# true results
y_true_original = np.hstack([
    scaler_Fx.inverse_transform(y_test.iloc[:, [0]]),
    scaler_Fy.inverse_transform(y_test.iloc[:, [1]]),
    scaler_Fz.inverse_transform(y_test.iloc[:, [2]]),
    scaler_Torque.inverse_transform(y_test.iloc[:, [3]]),
])
df_true = pd.DataFrame(y_true_original, columns=['Fx_true', 'Fy_true', 'Fz_true', 'Torque_true'])
# calculate relative error
true_values = df_true[['Fx_true', 'Fy_true', 'Fz_true', 'Torque_true']].values
pred_values = df_result[['Fx_pred', 'Fy_pred', 'Fz_pred', 'Torque_pred']].values
with np.errstate(divide='ignore', invalid='ignore'):
    relative_error = np.where(
        true_values != 0,
        (true_values - pred_values) * 100 / true_values,
        0.0
    )
df_relative_error = pd.DataFrame(relative_error, columns=['Fx_relative_error', 'Fy_relative_error', 'Fz_relative_error', 'Torque_relative_error'])
# merge all data into one dataframe
df_all = pd.concat([df_true, df_result, df_relative_error], axis=1)
df_all.to_excel('./data_for_train/predict_vs_true.xlsx', index=False)



    Fx_true   Fy_true   Fz_true  Torque_true   Fx_pred   Fy_pred   Fz_pred  \
0  4.258245 -0.023419  0.336380     0.073660  4.211762 -0.028422  0.332598   
1  2.817477  0.047223  0.276353     0.057315  2.844287  0.047866  0.280317   
2  7.505749  0.000337  0.560043     0.147226  7.532562 -0.001683  0.550125   
3  3.521817  0.070269  0.199701     0.055580  3.498842  0.072298  0.201026   
4  4.074701 -0.045790  0.348186     0.083170  4.123178 -0.043778  0.349795   

   Torque_pred  Fx_relative_error  Fy_relative_error  Fz_relative_error  \
0     0.074411           1.091605         -21.363071           1.124332   
1     0.057841          -0.951546          -1.361833          -1.434329   
2     0.147717          -0.357233         599.379334           1.770941   
3     0.055483           0.652379          -2.886427          -0.663257   
4     0.083757          -1.189706           4.395762          -0.462187   

   Torque_relative_error  
0              -1.019901  
1              -0.916239  

In [22]:
print(df_all.max())

Fx_true                     7.772109
Fy_true                     0.083090
Fz_true                     0.560043
Torque_true                 0.156934
Fx_pred                     7.857050
Fy_pred                     0.082378
Fz_pred                     0.550125
Torque_pred                 0.155104
Fx_relative_error           3.176112
Fy_relative_error        1274.359483
Fz_relative_error           5.095447
Torque_relative_error       3.368628
dtype: float64
